# **Homework 1: COVID-19 Cases Prediction (Regression)**

Objectives:
* Solve a regression problem with deep neural networks (DNN).
* Understand basic DNN training tips.
* Familiarize yourself with PyTorch.

If you have any questions, please contact the TAs via TA hours, NTU COOL, or email to mlta-2023-spring@googlegroups.com

In [31]:
!nvidia-smi

/bin/bash: /opt/conda/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Sun Oct 11 14:58:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             33W /   70W |     867MiB /  15360MiB |      0%      Default |
|                                         

# Import packages

In [32]:
# Numerical Operations
import math
import numpy as np

# Reading/Writing Data
import pandas as pd
import os
import csv

# For Progress Bar
from tqdm import tqdm

# Pytorch
import torch 
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

# For plotting learning curve
from torch.utils.tensorboard import SummaryWriter

# Download data
If the Google Drive links below do not work, you can use the dropbox link below or download data from [Kaggle](https://www.kaggle.com/t/a339b77fa5214978bfb8dde62d3151fe), and upload data manually to the workspace.

In [33]:
# google drive link
# !pip install gdown
# !gdown --id '1BjXalPZxq9mybPKNjF3h5L3NcF7XKTS-' --output covid_train.csv
# !gdown --id '1B55t74Jg2E5FCsKCsUEkPKIuqaY7UIi1' --output covid_test.csv

# dropbox link
!wget -O covid_train.csv https://www.dropbox.com/s/lmy1riadzoy0ahw/covid.train.csv?dl=0
!wget -O covid_test.csv https://www.dropbox.com/s/zalbw42lu4nmhr2/covid.test.csv?dl=0

/bin/bash: /opt/conda/lib/libtinfo.so.6: no version information available (required by /bin/bash)
--2026-10-11 14:58:58--  https://www.dropbox.com/s/lmy1riadzoy0ahw/covid.train.csv?dl=0
Resolving www.dropbox.com (www.dropbox.com)... 162.125.3.18, 2620:100:6018:18::a27d:312
Connecting to www.dropbox.com (www.dropbox.com)|162.125.3.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/ewl0ff7lviu0s7f53jp9o/covid.train.csv?rlkey=pocojbo26thh2ncv0xkxfafiv&dl=0 [following]
--2026-10-11 14:58:58--  https://www.dropbox.com/scl/fi/ewl0ff7lviu0s7f53jp9o/covid.train.csv?rlkey=pocojbo26thh2ncv0xkxfafiv&dl=0
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://ucdd072ab143941f2032d039c2d7.dl.dropboxusercontent.com/cd/0/inline/DJxhlWpbQ_pL2iHuZG4xFAoWwgRe_fVra3SO3GmPuFgC6imeyPciOjvS3vUe1SfxEdO88eFrbpUPiyNcti7BRG0s8kEjfjYIQIQ-6Aue6ejVXMzzd0yyrNXzvvbGWbZoPwLCSy4sugPszBuo9CoUVc

## Data describe 

In [34]:
# 1. 读取训练集
df = pd.read_csv('./covid_train.csv')

# 2. 查看数据规模（行数、列数）
print("数据集形状：", df.shape)

# 3. 查看所有列名
print("\n列名：")
print(df.columns.tolist())

# 4. 查看前 5 行数据
print("\n前 5 行数据：")
display(df.head())

数据集形状： (3009, 89)

列名：
['id', 'AL', 'AZ', 'CA', 'CO', 'CT', 'FL', 'GA', 'IL', 'IN', 'IA', 'KS', 'KY', 'LA', 'ME', 'MD', 'MA', 'MI', 'MN', 'MO', 'NJ', 'NM', 'NY', 'NC', 'OH', 'OK', 'OR', 'PA', 'SC', 'TN', 'TX', 'VA', 'WA', 'WV', 'WI', 'cli', 'ili', 'wnohh_cmnty_cli', 'wbelief_masking_effective', 'wbelief_distancing_effective', 'wcovid_vaccinated_friends', 'wlarge_event_indoors', 'wothers_masked_public', 'wothers_distanced_public', 'wshop_indoors', 'wrestaurant_indoors', 'wworried_catch_covid', 'hh_cmnty_cli', 'nohh_cmnty_cli', 'wearing_mask_7d', 'public_transit', 'worried_finances', 'tested_positive', 'cli.1', 'ili.1', 'wnohh_cmnty_cli.1', 'wbelief_masking_effective.1', 'wbelief_distancing_effective.1', 'wcovid_vaccinated_friends.1', 'wlarge_event_indoors.1', 'wothers_masked_public.1', 'wothers_distanced_public.1', 'wshop_indoors.1', 'wrestaurant_indoors.1', 'wworried_catch_covid.1', 'hh_cmnty_cli.1', 'nohh_cmnty_cli.1', 'wearing_mask_7d.1', 'public_transit.1', 'worried_finances.1', 'te

,id,AL,AZ,CA,CO,CT,FL,GA,IL,IN,...,wothers_distanced_public.2,wshop_indoors.2,wrestaurant_indoors.2,wworried_catch_covid.2,hh_cmnty_cli.2,nohh_cmnty_cli.2,wearing_mask_7d.2,public_transit.2,worried_finances.2,tested_positive.2
0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,19.271113,69.050180,38.102142,47.130223,22.686202,17.583283,62.925033,2.704414,39.222329,18.490787
1,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,19.500509,68.847156,37.338682,46.598421,22.484758,17.219515,62.771641,2.474973,41.209073,16.329253
2,2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,18.938706,68.694620,37.543537,46.858400,22.506261,17.128204,62.546116,2.569940,39.636816,16.522931
3,3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,18.698535,69.339191,37.751874,44.633652,22.369951,17.069263,61.517466,2.610086,38.926817,15.578501
4,4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,18.034980,69.564435,38.341833,44.100299,21.440588,16.207377,60.933647,2.790749,39.840306,14.171920


# Some Utility Functions

You do not need to modify this part.

In [35]:
def same_seed(seed): 
    '''Fixes random number generator seeds for reproducibility.'''
    torch.backends.cudnn.deterministic = True # PyTorch 的底层加速库 cuDNN，在计算时采用确定性算法
    torch.backends.cudnn.benchmark = False #关闭 cuDNN 的自动寻找最优卷积算法的功能
    np.random.seed(seed) # 固定 NumPy 库的随机种子
    torch.manual_seed(seed) # 固定 PyTorch CPU 操作的随机种子
    if torch.cuda.is_available(): # 固定所有可用 GPU 的随机种子
        torch.cuda.manual_seed_all(seed)

def train_valid_split(data_set, valid_ratio, seed): # 划分训练集和验证集
    '''Split provided training data into training set and validation set'''
    valid_set_size = int(valid_ratio * len(data_set)) 
    train_set_size = len(data_set) - valid_set_size
    train_set, valid_set = random_split(data_set, [train_set_size, valid_set_size], generator=torch.Generator().manual_seed(seed))
    return np.array(train_set), np.array(valid_set)

def predict(test_loader, model, device):
    model.eval() # Set your model to evaluation mode.
    preds = []
    for x in tqdm(test_loader):
        x = x.to(device)                        
        with torch.no_grad():                   
            pred = model(x)                     
            preds.append(pred.detach().cpu())   
    preds = torch.cat(preds, dim=0).numpy()  
    return preds

#### 1. `same_seed(seed)`：固定随机种子

#### 2. `train_valid_split(data_set, valid_ratio, seed)`：划分训练集和验证集
*   `random_split`：这是 PyTorch 提供的数据切分方法。它会根据计算好的大小随机抽取数据。这里非常巧妙地传入了一个临时的 `generator` 并固定了它的种子，这样既实现了随机切分，又保证了只要 `seed` 不变，每次切分出来的训练集和验证集都是完全一样的。

#### 3. `predict(test_loader, model, device)`：

*   `model.eval()`：**非常重要的一步**。将模型切换到“评估模式”。这会关闭 Dropout（随机丢弃神经元）和 Batch Normalization 的统计量更新，确保预测时模型结构是固定的。
*   `preds = []`：初始化一个空列表，用来存放每一批次的预测结果。
*   `for x in tqdm(test_loader):`：使用 `tqdm` 进度条遍历测试数据加载器。`x` 代表当前批次的数据（通常是一个 Tensor）。
*   `x = x.to(device)`：将当前批次的数据移动到指定的计算设备上（比如从 CPU 内存移动到 GPU 显存中）。
*   `with torch.no_grad():`：进入“不计算梯度”的上下文。在预测时我们不需要反向传播更新参数，关闭梯度计算可以**大幅节省显存并加快计算速度**。
*   `pred = model(x)`：将数据喂给模型，得到当前批次的预测输出。
*   `preds.append(pred.detach().cpu())`：
    *   `.detach()`：将预测结果从计算图中剥离出来（因为不需要梯度）。
    *   `.cpu()`：将结果从 GPU 移回 CPU（因为 NumPy 只能在 CPU 上处理）。
    *   最后将这一批结果存入列表。
*   `preds = torch.cat(preds, dim=0).numpy()`：
    *   `torch.cat(..., dim=0)`：将列表中所有批次的 Tensor 沿着第 0 维（样本维度）拼接成一个巨大的 Tensor。
    *   `.numpy()`：最后将拼接好的 Tensor 转换成 NumPy 数组，方便后续保存为 CSV 文件或进行其他分析。




# Dataset

In [36]:
class COVID19Dataset(Dataset):
    '''
    x: Features.
    y: Targets, if none, do prediction.
    '''
    def __init__(self, x, y=None):
        if y is None:
            self.y = y
        else:
            self.y = torch.FloatTensor(y)
        self.x = torch.FloatTensor(x)

    def __getitem__(self, idx):
        if self.y is None:
            return self.x[idx]
        else:
            return self.x[idx], self.y[idx]

    def __len__(self):
        return len(self.x)

# Neural Network Model
Try out different model architectures by modifying the class below.

In [37]:
class My_Model(nn.Module):
    def __init__(self, input_dim):
        super(My_Model, self).__init__()
        # TODO: modify model's structure, be aware of dimensions. 
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        x = self.layers(x)
        x = x.squeeze(1) # (B, 1) -> (B)
        return x

# Feature Selection
Choose features you deem useful by modifying the function below.

In [38]:
def select_feat(train_data, valid_data, test_data, select_all=True):
    '''Selects useful features to perform regression'''
    y_train, y_valid = train_data[:,-1], valid_data[:,-1]
    raw_x_train, raw_x_valid, raw_x_test = train_data[:,:-1], valid_data[:,:-1], test_data

    if select_all:
        feat_idx = list(range(raw_x_train.shape[1]))
    else:
        feat_idx = list(range(35, raw_x_train.shape[1])) # TODO: Select suitable feature columns.
        
    return raw_x_train[:,feat_idx], raw_x_valid[:,feat_idx], raw_x_test[:,feat_idx], y_train, y_valid

# Training Loop

In [41]:
def trainer(train_loader, valid_loader, model, config, device):

    criterion = nn.MSELoss(reduction='mean') # Define your loss function, do not modify this.

    # Define your optimization algorithm. 
    # TODO: Please check https://pytorch.org/docs/stable/optim.html to get more available algorithms.
    # TODO: L2 regularization (optimizer(weight decay...) or implement by your self).
    # optimizer = torch.optim.SGD(model.parameters(), lr=config['learning_rate'], momentum=0.7) 

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config['learning_rate'],
        weight_decay=1e-5
    )
    
    writer = SummaryWriter() # Writer of tensoboard.

    if not os.path.isdir('./models'):
        os.mkdir('./models') # Create directory of saving models.

    n_epochs, best_loss, step, early_stop_count = config['n_epochs'], math.inf, 0, 0

    for epoch in range(n_epochs):
        model.train() # Set your model to train mode.
        loss_record = []

        # tqdm is a package to visualize your training progress.
        # train_pbar = tqdm(train_loader, position=0, leave=True) 
        # 并不需要每次都看
        train_pbar = tqdm(train_loader, position=0, leave=False, disable=True)

        for x, y in train_pbar:
            optimizer.zero_grad()               # Set gradient to zero.
            x, y = x.to(device), y.to(device)   # Move your data to device. 
            pred = model(x)             
            loss = criterion(pred, y)
            loss.backward()                     # Compute gradient(backpropagation).
            optimizer.step()                    # Update parameters.
            step += 1
            loss_record.append(loss.detach().item())
            
            # Display current epoch number and loss on tqdm progress bar.
            train_pbar.set_description(f'Epoch [{epoch+1}/{n_epochs}]')
            train_pbar.set_postfix({'loss': loss.detach().item()})

        mean_train_loss = sum(loss_record)/len(loss_record)
        writer.add_scalar('Loss/train', mean_train_loss, step)

        model.eval() # Set your model to evaluation mode.
        loss_record = []
        for x, y in valid_loader:
            x, y = x.to(device), y.to(device)
            with torch.no_grad():
                pred = model(x)
                loss = criterion(pred, y)

            loss_record.append(loss.item())
            
        mean_valid_loss = sum(loss_record)/len(loss_record)
        # print(f'Epoch [{epoch+1}/{n_epochs}]: Train loss: {mean_train_loss:.4f}, Valid loss: {mean_valid_loss:.4f}')
        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{n_epochs}]: Train loss: {mean_train_loss:.4f}, Valid loss: {mean_valid_loss:.4f}, Best valid loss: {best_loss:.4f}')
        writer.add_scalar('Loss/valid', mean_valid_loss, step)

        if mean_valid_loss < best_loss:
            best_loss = mean_valid_loss
            torch.save(model.state_dict(), config['save_path']) # Save your best model
            print('Saving model with loss {:.3f}...'.format(best_loss))
            early_stop_count = 0
        else: 
            early_stop_count += 1

        
        
        if early_stop_count >= config['early_stop']:
            print('\nModel is not improving, so we halt the training session.')
            return


* AdamW 通常比基础 SGD 更容易训练；它会根据不同参数的梯度自适应调整学习率；weight_decay=1e-5 实现 L2 regularization，抑制模型过拟合；保留 config["learning_rate"]，方便后续调参。
* 修改为每10轮才打印一次

# Configurations
`config` contains hyper-parameters for training and the path to save your model.

In [40]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
config = {
    'seed': 5201314,      # Your seed number, you can pick your lucky number. :)
    'select_all': False,   # Whether to use all features.
    'valid_ratio': 0.2,   # validation_size = train_size * valid_ratio
    'n_epochs': 5000,     # Number of epochs.            
    'batch_size': 256, 
    'learning_rate': 1e-5,              
    'early_stop': 200,    # If model has not improved for this many consecutive epochs, stop training.     
    'save_path': './models/model.ckpt'  # Your model will be saved here.
}

* 把early_stop从600改为200：发现loss在epoch在1800左右就几乎不会降低了

# Dataloader
Read data from files and set up training, validation, and testing sets. You do not need to modify this part.

In [42]:
# Set seed for reproducibility
same_seed(config['seed'])


# train_data size: 3009 x 89 (35 states + 18 features x 3 days) 
# test_data size: 997 x 88 (without last day's positive rate)
train_data, test_data = pd.read_csv('./covid_train.csv').values, pd.read_csv('./covid_test.csv').values
train_data, valid_data = train_valid_split(train_data, config['valid_ratio'], config['seed'])

# Print out the data size.
print(f"""train_data size: {train_data.shape} 
valid_data size: {valid_data.shape} 
test_data size: {test_data.shape}""")

# Select features
x_train, x_valid, x_test, y_train, y_valid = select_feat(train_data, valid_data, test_data, config['select_all'])

# Print out the number of features.
print(f'number of features: {x_train.shape[1]}')

train_dataset, valid_dataset, test_dataset = COVID19Dataset(x_train, y_train), \
                                            COVID19Dataset(x_valid, y_valid), \
                                            COVID19Dataset(x_test)

# Pytorch data loader loads pytorch dataset into batches.
train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=config['batch_size'], shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False, pin_memory=True)

train_data size: (2408, 89) 
valid_data size: (601, 89) 
test_data size: (997, 88)
number of features: 53


# Start training!

In [43]:
model = My_Model(input_dim=x_train.shape[1]).to(device) # put your model and data on the same computation device.
trainer(train_loader, valid_loader, model, config, device)

Saving model with loss 543.322...
Saving model with loss 530.954...
Saving model with loss 508.748...
Saving model with loss 501.302...
Saving model with loss 468.691...
Saving model with loss 440.899...
Epoch [10/5000]: Train loss: 457.4529, Valid loss: 444.2392, Best valid loss: 440.8987
Saving model with loss 433.827...
Saving model with loss 412.706...
Saving model with loss 397.506...
Saving model with loss 388.958...
Saving model with loss 381.951...
Saving model with loss 370.208...
Saving model with loss 345.308...
Epoch [20/5000]: Train loss: 344.2801, Valid loss: 337.2656, Best valid loss: 345.3084
Saving model with loss 337.266...
Saving model with loss 336.111...
Saving model with loss 309.963...
Saving model with loss 299.494...
Saving model with loss 297.912...
Saving model with loss 294.202...
Saving model with loss 289.700...
Saving model with loss 284.378...
Saving model with loss 279.191...
Epoch [30/5000]: Train loss: 264.3174, Valid loss: 255.0919, Best valid loss: 

# Testing
The predictions of your model on testing set will be stored at `pred.csv`.

In [44]:
def save_pred(preds, file):
    ''' Save predictions to specified file '''
    with open(file, 'w') as fp:
        writer = csv.writer(fp)
        writer.writerow(['id', 'tested_positive'])
        for i, p in enumerate(preds):
            writer.writerow([i, p])

model = My_Model(input_dim=x_train.shape[1]).to(device)
model.load_state_dict(torch.load(config['save_path']))
preds = predict(test_loader, model, device) 
save_pred(preds, 'pred.csv')         

100%|██████████| 4/4 [00:00<00:00, 603.28it/s]


# Download

Run this block to download the `pred.csv` by clicking.

In [45]:
from IPython.display import FileLink
FileLink(r'pred.csv')

/kaggle/working/pred.csv

# Reference
This notebook uses code written by Heng-Jui Chang @ NTUEE (https://github.com/ga642381/ML2021-Spring/blob/main/HW01/HW01.ipynb)